# 04 — Gold: ознаки моделі для одного міста

Параметр `city`. Читає `workspace.silver.listings` і пише `workspace.gold.listing_features` (одна партиція на місто).

Тут лише **детерміновані перетворення рядка або міста**, без статистик з ціни: прапорці зручностей і щільність оголошень навколо.
Усе, що «вчиться» на даних (імпутація, рідкісні `property_type`, baseline), робиться в `05_train_model` і тільки на train.
Тому для нового міста цей ноутбук запускається без змін.

In [0]:
dbutils.widgets.text("city", "sicily")
city = dbutils.widgets.get("city").strip().lower()

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER = f"{CATALOG}.silver.listings"
FEATURES = f"{CATALOG}.gold.listing_features"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")


AMENITY_FLAGS = {
    "am_pool":         r"\bpool\b(?! table)",
    "am_sea":          r"beach access|waterfront|sea view|ocean view|beach view|bay view|harbor view|marina view|lake access",
    "am_ac":           r"air conditioning|\bac -",
    "am_hot_tub":      r"hot tub",
    "am_elevator":     r"\belevator\b",
    "am_free_parking": r"free (?:driveway |street )?parking|free parking",
    "am_dishwasher":   r"dishwasher",
    "am_workspace":    r"dedicated workspace",
}

DENSITY_STEPS = {"log_density_1km": 0.005, "log_density_5km": 0.02}

In [0]:
s = spark.table(SILVER).filter(F.col("city") == city)
if s.limit(1).count() == 0:
    raise ValueError(f"У {SILVER} немає рядків для city='{city}' — спершу запустіть bronze і silver")

am = F.lower(F.coalesce(F.col("amenities"), F.lit("")))
f = s
for name, pattern in AMENITY_FLAGS.items():
    f = f.withColumn(name, am.rlike(pattern))

def add_density(df, out_col, step):
    """Скільки оголошень цього міста в 3×3 клітинках навколо (рахуємо всі оголошення, ціна не потрібна)."""
    cell = lambda: [F.floor(F.col("latitude") / step).alias("gy"), F.floor(F.col("longitude") / step).alias("gx")]
    counts = df.filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull()).select(*cell()).groupBy("gy", "gx").count()
    offsets = spark.createDataFrame([(dy, dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1)], "dy int, dx int")
    window = (counts.crossJoin(offsets)
              .select((F.col("gy") - F.col("dy")).alias("gy"), (F.col("gx") - F.col("dx")).alias("gx"), "count")
              .groupBy("gy", "gx").agg(F.sum("count").alias("_n")))
    return (df.select("*", *cell()).join(window, ["gy", "gx"], "left")
              .withColumn(out_col, F.when(F.col("latitude").isNotNull(), F.log1p(F.coalesce(F.col("_n"), F.lit(0)).cast("double"))))
              .drop("gy", "gx", "_n"))

for col, step in DENSITY_STEPS.items():
    f = add_density(f, col, step)

In [0]:
KEEP = [
    "city", "snapshot_date", "id", "host_id", "name", "listing_url", "neighbourhood_cleansed",
    "latitude", "longitude",
    "price", "log_price", "has_price", "outlier_class",
    "room_type", "property_type", "accommodates", "bedrooms", "beds", "bathrooms", "bathrooms_shared",
    "amenities_count", "calculated_host_listings_count_entire_homes", "calculated_host_listings_count_private_rooms",
    "host_tenure_months", "long_stay", "min_nights_missing", "has_license",
    *AMENITY_FLAGS, *DENSITY_STEPS,
]
gold = (f.select(*KEEP)
         .withColumn("in_model_scope", F.col("has_price") & F.col("outlier_class").isin("normal", "tail"))
         .withColumn("_built_at", F.current_timestamp()))

w = gold.write.format("delta").option("mergeSchema", "true")
if spark.catalog.tableExists(FEATURES):
    w.mode("overwrite").option("replaceWhere", f"city = '{city}'").saveAsTable(FEATURES)
else:
    w.mode("overwrite").partitionBy("city").saveAsTable(FEATURES)

In [0]:
out = spark.table(FEATURES).filter(F.col("city") == city)
assert out.count() == s.count(), "кількість рядків у gold не дорівнює silver"
display(out.agg(
    F.count("*").alias("rows"), F.sum(F.col("in_model_scope").cast("int")).alias("in_model_scope"),
    *[F.round(F.avg(F.col(c).cast("int")), 3).alias(c) for c in AMENITY_FLAGS],
    *[F.round(F.avg(c), 2).alias(c) for c in DENSITY_STEPS]))